In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🔌 Circuit Breakers & Resilience Patterns Guide

**Building fault-tolerant systems with circuit breakers, retries, timeouts, and bulkheads**

This guide covers:
- Circuit breaker pattern (open, closed, half-open states)
- Retry strategies (exponential backoff, jitter)
- Timeout and deadline management
- Bulkhead isolation pattern
- Rate limiting and backpressure
- Graceful degradation
- Health checks and probes
- Resilience frameworks (Polly, Resilience4j)
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Circuit Breaker Pattern

### Manual Circuit Breaker Implementation
```python
from enum import Enum
from datetime import datetime, timedelta
import time

class CircuitState(Enum):
    CLOSED = "closed"      # Normal operation
    OPEN = "open"          # Failing, reject requests
    HALF_OPEN = "half_open"  # Testing recovery

class CircuitBreaker:
    """Circuit breaker for fault tolerance"""
    
    def __init__(
        self,
        failure_threshold: int = 5,
        recovery_timeout: int = 60,
        expected_exception: Exception = Exception
    ):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.expected_exception = expected_exception
        
        self.failure_count = 0
        self.success_count = 0
        self.last_failure_time = None
        self.state = CircuitState.CLOSED
    
    def call(self, func, *args, **kwargs):
        """Execute function through circuit breaker"""
        
        if self.state == CircuitState.OPEN:
            if self._should_attempt_reset():
                self.state = CircuitState.HALF_OPEN
                self.success_count = 0
            else:
                raise Exception(f"Circuit breaker open (failures: {self.failure_count})")
        
        try:
            result = func(*args, **kwargs)
            self._on_success()
            return result
        
        except self.expected_exception as e:
            self._on_failure()
            raise
    
    def _on_success(self):
        """Handle successful call"""
        self.failure_count = 0
        
        if self.state == CircuitState.HALF_OPEN:
            self.success_count += 1
            
            if self.success_count >= 2:
                self.state = CircuitState.CLOSED
                print("Circuit breaker closed - service recovered")
    
    def _on_failure(self):
        """Handle failed call"""
        self.failure_count += 1
        self.last_failure_time = datetime.now()
        
        if self.failure_count >= self.failure_threshold:
            self.state = CircuitState.OPEN
            print(f"Circuit breaker open - too many failures ({self.failure_count})")
    
    def _should_attempt_reset(self) -> bool:
        """Check if recovery timeout has elapsed"""
        if self.last_failure_time is None:
            return False
        
        elapsed = (datetime.now() - self.last_failure_time).total_seconds()
        return elapsed >= self.recovery_timeout
    
    def get_state(self) -> str:
        return self.state.value

# Usage
breaker = CircuitBreaker(failure_threshold=3, recovery_timeout=30)

for i in range(10):
    try:
        result = breaker.call(risky_api_call, param1, param2)
        print(f"Success: {result}")
    except Exception as e:
        print(f"Error: {str(e)}")
    
    time.sleep(2)
```

### Pybreaker Library Example
```python
from pybreaker import CircuitBreaker as PyCircuitBreaker

class APIClient:
    def __init__(self):
        self.breaker = PyCircuitBreaker(
            fail_max=5,
            reset_timeout=60,
            exclude=[KeyError, ValueError],  # Don't count these
            listeners=[self._on_state_change]
        )
    
    @property
    def breaker_decorator(self):
        return self.breaker
    
    def _on_state_change(self, cb, before, after):
        print(f"Circuit breaker state changed: {before} -> {after}")
    
    @PyCircuitBreaker(fail_max=5, reset_timeout=60)
    def call_external_api(self, endpoint: str):
        """Protected API call"""
        response = requests.get(f"https://api.example.com/{endpoint}")
        return response.json()
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Retry Strategies

### Exponential Backoff with Jitter
```python
import random
import time

class RetryStrategy:
    """Configurable retry strategies"""
    
    @staticmethod
    def exponential_backoff(attempt: int, base_delay: float = 1.0, max_delay: float = 60.0):
        """Calculate delay with exponential backoff"""
        delay = min(base_delay * (2 ** attempt), max_delay)
        return delay
    
    @staticmethod
    def exponential_backoff_with_jitter(attempt: int, base_delay: float = 1.0):
        """Add jitter to avoid thundering herd"""
        delay = base_delay * (2 ** attempt)
        jitter = random.uniform(0, delay)
        return jitter
    
    @staticmethod
    def linear_backoff(attempt: int, step: float = 1.0, max_delay: float = 60.0):
        """Linear backoff"""
        return min(step * attempt, max_delay)

class RetryableRequest:
    """Wrapper for retryable HTTP requests"""
    
    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries
    
    def execute(self, func, *args, **kwargs):
        """Execute function with retries"""
        
        for attempt in range(self.max_retries + 1):
            try:
                return func(*args, **kwargs)
            
            except Exception as e:
                if attempt == self.max_retries:
                    raise
                
                # Calculate backoff delay
                delay = RetryStrategy.exponential_backoff_with_jitter(attempt)
                
                print(f"Attempt {attempt + 1} failed: {str(e)}. Retrying in {delay:.2f}s")
                time.sleep(delay)

# Usage
retrier = RetryableRequest(max_retries=3)
result = retrier.execute(requests.get, "https://api.example.com/data")
```

### Decorators for Retries
```python
import functools

def retry(max_attempts: int = 3, backoff: str = 'exponential'):
    """Decorator for retry logic"""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_attempts - 1:
                        raise
                    
                    if backoff == 'exponential':
                        delay = RetryStrategy.exponential_backoff(attempt)
                    else:
                        delay = RetryStrategy.linear_backoff(attempt)
                    
                    print(f"{func.__name__} failed (attempt {attempt + 1}). Retrying in {delay}s")
                    time.sleep(delay)
        
        return wrapper
    return decorator

@retry(max_attempts=3, backoff='exponential')
def fetch_user_data(user_id: str):
    """Automatically retried function"""
    response = requests.get(f"/api/users/{user_id}")
    return response.json()
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Bulkhead Pattern

### Thread Pool Isolation
```python
from concurrent.futures import ThreadPoolExecutor
from threading import Lock

class BulkheadIsolation:
    """Bulkhead pattern for thread isolation"""
    
    def __init__(self, pool_size: int = 10):
        self.thread_pool = ThreadPoolExecutor(max_workers=pool_size)
        self.lock = Lock()
    
    def execute_isolated(self, func, *args, **kwargs):
        """Execute in isolated thread pool"""
        future = self.thread_pool.submit(func, *args, **kwargs)
        return future.result(timeout=5)  # 5-second timeout

class ServiceClient:
    """Service with multiple bulkheads"""
    
    def __init__(self):
        self.primary_pool = ThreadPoolExecutor(max_workers=5)
        self.secondary_pool = ThreadPoolExecutor(max_workers=3)
    
    def call_primary_service(self):
        """Isolated thread pool for primary service"""
        return self.primary_pool.submit(self._primary_call)
    
    def call_secondary_service(self):
        """Isolated thread pool for secondary service"""
        return self.secondary_pool.submit(self._secondary_call)
    
    def _primary_call(self):
        # Primary service logic
        pass
    
    def _secondary_call(self):
        # Secondary service logic
        pass
```

### Semaphore-based Bulkhead
```python
from threading import Semaphore

class SemaphoreBulkhead:
    """Use semaphores to limit concurrent requests"""
    
    def __init__(self, max_concurrent: int = 10):
        self.semaphore = Semaphore(max_concurrent)
    
    def execute(self, func, *args, **kwargs):
        """Execute with concurrency limit"""
        
        if not self.semaphore.acquire(blocking=False):
            raise Exception("Bulkhead limit exceeded")
        
        try:
            return func(*args, **kwargs)
        finally:
            self.semaphore.release()
```
</MySCode.Cell>

<MySCode.Cell language="markdown">
## 4. Timeout & Deadline Management

### Timeout Implementation
```python
import signal

class TimeoutError(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutError("Operation timed out")

class TimeoutManager:
    """Manage operation timeouts"""
    
    @staticmethod
    def execute_with_timeout(func, timeout_seconds: int, *args, **kwargs):
        """Execute function with timeout"""
        
        # Set timeout signal
        signal.signal(signal.SIGALRM, timeout_handler)
        signal.alarm(timeout_seconds)
        
        try:
            result = func(*args, **kwargs)
            signal.alarm(0)  # Cancel alarm
            return result
        except TimeoutError:
            print(f"Operation timed out after {timeout_seconds}s")
            raise
        finally:
            signal.alarm(0)

class Deadline:
    """Deadline for cascading timeouts"""
    
    def __init__(self, deadline_seconds: int):
        self.deadline = time.time() + deadline_seconds
    
    def remaining_time(self) -> float:
        """Get remaining time"""
        remaining = self.deadline - time.time()
        return max(0, remaining)
    
    def is_exceeded(self) -> bool:
        """Check if deadline exceeded"""
        return self.remaining_time() <= 0
```

### Cascading Timeouts
```python
class CascadingService:
    """Service with deadline propagation"""
    
    def process_request(self, request, deadline_seconds: int = 10):
        """Process with cascading timeout"""
        deadline = Deadline(deadline_seconds)
        
        # Call service 1
        result1 = self._call_service_1(deadline)
        
        # Call service 2 with remaining time
        remaining = deadline.remaining_time()
        result2 = self._call_service_2(result1, remaining)
        
        # Call service 3 with remaining time
        remaining = deadline.remaining_time()
        result3 = self._call_service_3(result2, remaining)
        
        return result3
    
    def _call_service_1(self, deadline):
        if deadline.is_exceeded():
            raise TimeoutError("Deadline exceeded")
        time.sleep(2)
        return "result1"
    
    def _call_service_2(self, data, timeout):
        time.sleep(1)
        return "result2"
    
    def _call_service_3(self, data, timeout):
        time.sleep(1)
        return "result3"
```
</MySCode.Cell>

<MySCode.Cell language="markdown">
## 5. Resilience Patterns Checklist

✅ **Circuit Breaker**
- [ ] Failure threshold configured
- [ ] State transitions implemented
- [ ] Recovery timeout set
- [ ] Monitoring configured
- [ ] Fallback strategy defined

✅ **Retries**
- [ ] Retry strategy chosen (exponential, linear)
- [ ] Jitter implemented
- [ ] Max attempts limited
- [ ] Idempotency ensured
- [ ] Backoff measured

✅ **Bulkheads**
- [ ] Thread pools isolated
- [ ] Pool sizes configured
- [ ] Queue limits set
- [ ] Rejection policy defined
- [ ] Monitoring enabled

✅ **Timeouts**
- [ ] Timeout values set for each operation
- [ ] Deadline propagation implemented
- [ ] Cascading timeouts working
- [ ] Graceful degradation on timeout
- [ ] Alerts configured

✅ **Monitoring**
- [ ] Metrics collected (failures, latency, throughput)
- [ ] Dashboards created
- [ ] Alerts configured
- [ ] Logs capture resilience events
- [ ] Performance impact measured
</MySCode.Cell>
```